# Initialize 

In [1]:
from pathlib import Path
import numpy as np
import geopandas as gpd
import xarray as xr
from tqdm.notebook import tqdm

# local modules
import model_chain_inference.generate_data as generate_data
import model_chain_inference.model_core as model
import model_chain_inference.bernstein as bernstein
import chaintools.tools_grid as tgrid
import chaintools.tools_geometry as tgeo

# Data import

In [2]:
path = Path("../data")
grid_data_file = path / "grid_data_flat.h5"
event_data_file = path / "event_data.h5"
fault_data_file = path / "fault_data.h5"

In [3]:
polygon_data = generate_data.get_polygon_gdf(path)
fault_data = generate_data.get_faults(path)
event_data = generate_data.get_catalogue()

In [4]:
grid_data = generate_data.get_grid(path)

# CONVERT for backwards compatibility
grid_data = grid_data.sel(cm_grid_version="cm_NAM2018")
grid_data["compressibility"] = (
    grid_data["compressibility"] / 10.0
)  # convert from MPa^-1 to 1/bar
grid_data["compressibility"].attrs["units"] = "1/bar"

# Experiment settings

In [5]:
bernstein_degrees = [4]

Following parameters are discretized model parameters to be interpolated during model calibration.

In [ ]:
parameters = xr.Dataset(
    {
        "rmax": np.linspace(0.375, 0.525, 7),
        "sigma": np.linspace(2200, 3800, 9),
    }
)
parameters["hs_exp"] = grid_data["hs_exp"]
parameters["M_plastic"] = grid_data["M_plastic"]

# Grid data curation

Determine a common area where data is present.
We consider grid data as constant in each cell, represented by the center of the cell.

In [52]:
grid_data["support"] = (
    xr.full_like(grid_data["x"] * grid_data["y"], 1)
    .where(grid_data["compressibility"] > 0.0, 0)
    .where(grid_data["reservoir_thickness"] > 0.0, 0)
    .where(np.isfinite(grid_data["reservoir_thickness"]), 0)
    .where(grid_data["pressure_drop"].isel({"datetime": -1}) > 0.0, 0)
)

# Prepare polygons

In [53]:
grid_support_polygon = tgeo.get_grid_support_polygon(grid_data["support"], 500.0)
polygons = xr.concat(
    [
        xr.DataArray(polygon_data["geometry"], dims="polygon").astype("object"),
        xr.DataArray(grid_support_polygon, coords={"polygon": "grid_support"}),
    ],
    dim="polygon",
)

In [54]:
gpd_pols = gpd.GeoDataFrame(polygons.to_dataframe(), crs="EPSG:28992")
gpd_pols.to_file(path / "groningen_polygons.shp", driver="ESRI Shapefile")

# Interpolate grid data to faults

In [55]:
fault_data = fault_data.merge(
    grid_data.interp(
        {
            "x": fault_data["x"],
            "y": fault_data["y"],
        },
        method="nearest",
    ).drop_vars(["x", "y"]),
    compat="override",
)

In [56]:
fault_filter = fault_data["support"] > 0.0
fault_data = fault_data.sel(ID=fault_filter)

In [ ]:
# there is a difference between the thickness inferred from the faults and the
# thickness from the grid
# the thickness in the faults seems to be noisy.. while the grid is smooth
# we should go with grid, although historically, B&O have been using the fault
# thickness
extended_reservoir_thickness = xr.Dataset(
    {
        "grid": fault_data["reservoir_thickness"],
        "faults": fault_data["ThicknessTotal"],
    },
).to_array("thickness_origin")
fault_data.update(
    xr.Dataset(
        {
            "reservoir_thickness": extended_reservoir_thickness,
        }
    ),
)

# Add fault attributes

In [58]:
fault_attributes = {
    "azimuth": np.fmod(fault_data["Azimuth"] + 90.0, 180.0),
    "throw_clipped": np.minimum(fault_data["Throw"], fault_data["reservoir_thickness"]),
    "throw_thickness_ratio": fault_data["Throw"] / fault_data["reservoir_thickness"],
    "fault_area": fault_data["reservoir_thickness"] * fault_data["StrikeLength"],
}

# Map fault data to grid

## Weights and burn-in for rmax-models

In [59]:
step = xr.DataArray(
    [grid_data[d].diff(d).mean().values for d in ["x", "y"]], dims=["loc"]
)

In [60]:
fault_attributes["weights_rmax"] = (
    fault_attributes["throw_thickness_ratio"] < (parameters["rmax"] / 2)
).astype(float)

In [61]:
fault_attributes["fault_area_rmax"] = (
    fault_attributes["fault_area"] * fault_attributes["weights_rmax"]
)

In [62]:
fault_data.update(xr.Dataset(fault_attributes))

In [63]:
throw_max_rmax = tgrid.aggregate_to_grid(
    fault_data[["x", "y"]],
    step,
    fault_data["weights_rmax"] * fault_data["Throw"],
    marginalize_dims=["ID"],
    operator=np.fmax,  # NAM aproach to assign max gradient to each cell
    order=0,
)  # max throw
throw_sum_rmax = tgrid.aggregate_to_grid(
    fault_data[["x", "y"]],
    step,
    fault_data["weights_rmax"] * fault_data["Throw"] * fault_data["StrikeLength"],
    marginalize_dims=["ID"],
    operator=np.add,  # default, actually
    order=0,
)  # extruding fault area
fault_length_rmax = tgrid.aggregate_to_grid(
    fault_data[["x", "y"]],
    step,
    fault_data["weights_rmax"] * fault_data["StrikeLength"],
    marginalize_dims=["ID"],
    operator=np.add,  # default, actually
    order=0,
)  # fault length
fault_area_rmax = tgrid.aggregate_to_grid(
    fault_data[["x", "y"]],
    step,
    fault_data["weights_rmax"] * fault_data["fault_area"],
    marginalize_dims=["ID"],
    operator=np.add,  # default, actually
    order=0,
)  # fault area
throw_average_rmax = throw_sum_rmax / fault_length_rmax

In [64]:
grid_data["throw_max_rmax"] = throw_max_rmax
grid_data["throw_average_rmax"] = throw_average_rmax
grid_data["fault_length_rmax"] = fault_length_rmax
grid_data["fault_area_rmax"] = fault_area_rmax

## Weights and burn-in for Bernstein-decomposition models

In [65]:
# for the Bernstein weighting we use r based on the map thickness
w_bernstein = [
    bernstein.bernstein_partition(
        fault_data["throw_thickness_ratio"],
        degree=n,
        fractiles=True,
        dim="ID",
    )
    for n in bernstein_degrees
]
fault_attributes = {
    "weights_bs": xr.concat(w_bernstein, dim="bernstein_basis"),
}
fault_attributes["fault_area_bs"] = (
    fault_data["fault_area"] * fault_attributes["weights_bs"]
)
fault_data.update(
    xr.Dataset(fault_attributes),
)

In [66]:
# throw_max does not appear to make sense when using Bernstein decomposition
# the average throw in grid cell is weighted by the fault length
throw_sum_bs = tgrid.aggregate_to_grid(
    samples=fault_data[["x", "y"]],
    target_step=step,
    weights=fault_attributes["weights_bs"]
    * fault_data["Throw"]
    * fault_data["StrikeLength"],
    marginalize_dims=["ID"],
    order=0,
)
fault_length_bs = tgrid.aggregate_to_grid(
    samples=fault_data[["x", "y"]],
    target_step=step,
    weights=fault_attributes["weights_bs"] * fault_data["StrikeLength"],
    marginalize_dims=["ID"],
    order=0,
)
fault_area_bs = tgrid.aggregate_to_grid(
    samples=fault_data[["x", "y"]],
    target_step=step,
    weights=fault_attributes["weights_bs"] * fault_data["fault_area"],
    marginalize_dims=["ID"],
    order=0,
)
throw_average_bs = throw_sum_bs / fault_length_bs

In [67]:
grid_attributes = {
    "throw_average_bs": throw_average_bs,
    "fault_length_bs": fault_length_bs,
    "fault_area_bs": fault_area_bs,
}
grid_data.update(xr.Dataset(grid_attributes))

In [68]:
# fancy stuff: for each degree calculate the total fault length separately
fault_total_length = (
    fault_length_bs.groupby("bernstein_degree")
    .sum()
    .rename({"bernstein_degree": "bd"})
    .sel({"bd": fault_length_bs["bernstein_degree"]})
    .drop_vars("bd")
)
bernstein_fraction = fault_length_bs / fault_total_length
grid_data.update(xr.Dataset({"bernstein_fraction": bernstein_fraction}))

# Stress calculations

Stresses are calculated using the full spatial extent of the input data, i.e. not filtered by the buffered gas field polygon.

### Linear

In [69]:
grid_data["pressure_drop_nondecreasing"] = np.clip(
    tgrid.xr_make_nondecreasing(grid_data["pressure_drop"], dim="datetime"), 0.0, None
)

In [70]:
grid_data["stress_linear"] = model.incremental_stress(
    pressure_drop=grid_data["pressure_drop_nondecreasing"],
    gradient=1.0,
    poisson_ratio=0.2,
    bulk_modulus=1.0 / grid_data["compressibility"],
    solid_modulus=10 ** parameters["hs_exp"],
)

In [71]:
grid_data["stress_linear_rmax"] = model.incremental_stress(
    pressure_drop=grid_data["pressure_drop_nondecreasing"],
    gradient=grid_data["throw_average_rmax"],
    poisson_ratio=0.2,
    bulk_modulus=1.0 / grid_data["compressibility"],
    solid_modulus=10 ** parameters["hs_exp"],
)

### RTiCM

In [72]:
grid_data["stress_RTiCM"] = tgrid.xr_make_nondecreasing(
    grid_data["horizontal_stress_RTiCM"],
    dim="datetime",
)
grid_data["stress_RTiCM_rmax"] = (
    grid_data["stress_RTiCM"] * grid_data["throw_average_rmax"]
)

### Linear - Bernstein

In [73]:
grid_attributes = {}
grid_attributes["stress_linear_bs"] = model.incremental_stress(
    pressure_drop=grid_data["pressure_drop_nondecreasing"],
    gradient=grid_data["throw_average_bs"],
    poisson_ratio=0.2,
    bulk_modulus=1.0 / grid_data["compressibility"],
    solid_modulus=10 ** parameters["hs_exp"],
)
grid_data.update(xr.Dataset(grid_attributes))

### RTiCM - Bernstein

In [74]:
grid_attributes = {}
grid_attributes["stress_RTiCM_bs"] = (
    tgrid.xr_make_nondecreasing(
        grid_data["horizontal_stress_RTiCM"],
        dim="datetime",
    )
    * grid_data["throw_average_bs"]
)
grid_data.update(xr.Dataset(grid_attributes))

In [75]:
fault_attributes = {}
for var in ["stress_RTiCM", "stress_linear"]:
    fault_attributes[var + "_throw"] = (
        grid_data[var]
        .interp(x=fault_data["x"], y=fault_data["y"], method="nearest")
        .drop_vars(["x", "y"])
    ) * fault_data["throw_clipped"]
fault_data.update(xr.Dataset(fault_attributes))

# Incorporate polygon data

In [76]:
grid_spacing = np.mean([grid_data[d].diff(d).mean().item() for d in ["x", "y"]])

In [77]:
grid_attributes = {}
grid_attributes["overlap_fraction"] = tgeo.xr_cell_polygon_overlap_fraction(
    grid_data,
    polygons,
    0.5 * grid_spacing,
    "square",
)
grid_attributes["support_fraction"] = (
    grid_attributes["overlap_fraction"] * grid_data["support"]
)
grid_data.update(xr.Dataset(grid_attributes))

In [78]:
event_data["polygon_distance"] = tgeo.xr_point_polygon_distance(
    event_data[["x", "y"]], polygons
)

# Make lean flat grid

Use only the grid points that are within the support of the input data for following calculations

In [79]:
grid_data_flat = grid_data.stack({"loc": ["x", "y"]})

In [80]:
grid_data_flat = grid_data_flat.sel({"loc": grid_data_flat["support"] > 0})

In [81]:
flat_nodes = grid_data_flat["loc"]

# Persistance

In [ ]:
event_data.to_netcdf(event_data_file, mode="w")

In [ ]:
fault_data.drop_vars("geometry").reset_index(["ID", "bernstein_basis"]).to_netcdf(
    fault_data_file, mode="w"
)

In [ ]:
grid_data_flat.reset_index(["loc", "bernstein_basis"]).to_netcdf(
    grid_data_file, mode="w"
)

# Smooth and save

In [ ]:
with xr.open_dataset(grid_data_file, decode_coords="all", decode_timedelta=False) as ds:
    grid_data_vars = list(ds.data_vars)

In [ ]:
grid_attributes = {}
for key in tqdm([k for k in grid_data_vars if k.startswith("fault_")]):
    print(key)
    with xr.open_dataset(
        grid_data_file, decode_coords="all", decode_timedelta=False
    ) as ds:
        data_flat = ds[key].load()

    if "bernstein_degree" in data_flat.coords:
        data_flat = data_flat.set_xindex(["bernstein_degree", "bernstein_index"])

    data = data_flat.set_xindex(["x", "y"]).unstack("loc")

    smooth_data = (
        tgrid.xr_smooth(
            data,
            parameters["sigma"],
        )
        .stack({"loc": ["x", "y"]})
        .sel({"loc": flat_nodes})
    )
    smooth_data.name = key + "_smooth"
    smooth_data.reset_index(
        [v for v in ["loc", "bernstein_basis"] if v in smooth_data.dims]
    ).to_netcdf(grid_data_file, mode="a")
    grid_attributes[key + "_smooth"] = smooth_data

  0%|          | 0/4 [00:00<?, ?it/s]

fault_length_rmax
fault_area_rmax
fault_area_rmax
fault_length_bs
fault_length_bs
fault_area_bs
fault_area_bs


In [87]:
grid_data_flat.update(xr.Dataset(grid_attributes))

In [88]:
for key in tqdm([k for k in grid_data_vars if k.startswith("stress")]):
    print(key)
    with xr.open_dataset(
        grid_data_file, decode_coords="all", decode_timedelta=False
    ) as ds:
        data_flat = ds[key].load()
        if "bs" in key and "stress" in key:
            bf_flat = ds["bernstein_fraction"].load()
        else:
            bf_flat = None

    if "bernstein_degree" in data_flat.coords:
        data_flat = data_flat.set_xindex(["bernstein_degree", "bernstein_index"])
    data = data_flat.set_xindex(["x", "y"]).unstack("loc")

    if bf_flat is not None:
        if "bernstein_degree" in bf_flat.coords:
            bf_flat = bf_flat.set_xindex(["bernstein_degree", "bernstein_index"])
        bf = bf_flat.set_xindex(["x", "y"]).unstack("loc")
        data = data * bf

    ignore_nans = True if "fakeNAM" in key else False
    smooth_data = (
        tgrid.xr_smooth(
            data,
            parameters["sigma"],
            ignore_nans=ignore_nans,
        )
        .stack({"loc": ["x", "y"]})
        .sel({"loc": flat_nodes})
    )
    smooth_data.name = key + "_smooth"
    smooth_data.reset_index(
        [v for v in ["loc", "bernstein_basis"] if v in smooth_data.dims]
    ).to_netcdf(grid_data_file, mode="a")

  0%|          | 0/6 [00:00<?, ?it/s]

stress_linear
stress_linear_rmax
stress_linear_rmax
stress_RTiCM
stress_RTiCM
stress_RTiCM_rmax
stress_RTiCM_rmax
stress_linear_bs
stress_linear_bs
stress_RTiCM_bs
